In [11]:
import numpy as np
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

def load_tokenizer_and_model(model_name_or_path):
    return (GPT2Tokenizer.from_pretrained(model_name_or_path),
            GPT2LMHeadModel.from_pretrained(model_name_or_path))

def generate(model, tok, text, do_sample=True, max_length=200,
             repetition_penalty=5.0, top_k=5, top_p=0.95,
             temperature=1.0, num_beams=1, no_repeat_ngram_size=3):
    input_ids = tok.encode(text, return_tensors="pt")
    try:
        print(model.generate.__globals__['__file__'])
    except Exception:
        pass
    out = model.generate(
        input_ids,
        max_length=max_length,
        repetition_penalty=repetition_penalty,
        do_sample=do_sample,
        top_k=top_k, top_p=top_p, temperature=temperature,
        num_beams=num_beams, no_repeat_ngram_size=no_repeat_ngram_size,
        pad_token_id=tok.eos_token_id,   # чтобы не было warning
    )
    return [tok.decode(o, skip_special_tokens=True,
                       clean_up_tokenization_spaces=False) for o in out]

In [12]:
tok, model = load_tokenizer_and_model("sberbank-ai/rugpt3large_based_on_gpt2")
model.eval()

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: sberbank-ai/rugpt3large_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1536)
    (wpe): Embedding(2048, 1536)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-23): 24 x GPT2Block(
        (ln_1): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=4608, nx=1536)
          (c_proj): Conv1D(nf=1536, nx=1536)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=6144, nx=1536)
          (c_proj): Conv1D(nf=1536, nx=6144)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1536, out_features=50257, bias=False)
)

In [13]:
TARGET = "машина"

PROMPT = "Однажды утром на дороге появилась новая "

generated = generate(model, tok, PROMPT, do_sample=True, max_length=200)
text = generated[0]
print("\nТекст: \n", text)

if TARGET.lower() in text.lower():
    print(f"\nСлово «{TARGET}» найдено")
else:
    print(f"\nНЕТ слова")

/usr/local/lib/python3.13/dist-packages/torch/utils/_contextlib.py

Текст: 
 Однажды утром на дороге появилась новая  погоня.
Сержант и его напарник, сержант-связист были в засаде у дороги с биноклями через оптику: они видели приближающуюся к ним машину со стороны леса по которой шла стрельба из пулемета «максим». Они заметили ее издалека - она была очень большая (больше грузовика). В бинокль было видно как от машины отъехала еще одна машина , которая остановилась рядом . На этот раз это был грузовик ГАЗ 66 – за рулем сидел человек лет пятидесяти пяти(этот тип людей не любит больших машин ) Он вышел покурить перед тем самым местом где прятался Сержанты решили что он едет их убивать ! 
 Но тот остановился недалеко до них ... Снял фуражку : "Я ваш новый русский царь" И пошел дальше! Когда же вернулся назад увидел своих преследователей сидящих сзади !!! А когда подошел ближе понял почему так произошло … У этого человека под правой рукой висела кобура пистолета Макарова а левой руке пистол